In [1]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [3]:
SEQUENCE_FEATURES = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "user_transaction_count_before",
    "previous_transaction_amount",
    "time_since_previous_transaction",
    "previous_average_amount",
    "amount_deviation",
    "amount_to_previous_average",
    "balance_depletion",
    "amount_to_balance_ratio",
    "receiver_transaction_count_before",
    "receiver_previous_amount",
    "receiver_transaction_frequency",
    "user_transfer_count_before",
    "user_cashout_count_before",
    "transaction_velocity"
]

print("GRU features:", len(SEQUENCE_FEATURES))

GRU features: 19


In [4]:
SEQUENCE_LENGTH = 5

print("Sequence length:", SEQUENCE_LENGTH)

Sequence length: 5


In [5]:
TRAIN_FILE = "../data/processed/train_data.csv"

train_sample = pd.read_csv(
    TRAIN_FILE,
    nrows=100_000
)

print("Sample shape:", train_sample.shape)

Sample shape: (100000, 25)


In [6]:
train_sample = train_sample.sort_values(
    ["nameOrig", "step"]
).reset_index(drop=True)

In [7]:
def create_sequences(
    df,
    feature_columns,
    sequence_length=5
):
    
    X_sequences = []
    y_labels = []

    for user, user_df in df.groupby("nameOrig", sort=False):

        user_df = user_df.sort_values("step")

        feature_values = (
            user_df[feature_columns]
            .fillna(0)
            .values
        )

        labels = user_df["isFraud"].values

        for i in range(len(user_df)):

            start = max(
                0,
                i - sequence_length
            )

            sequence = feature_values[start:i]

            # Need at least one previous transaction
            if len(sequence) == 0:
                continue

            # Left-pad with zeros
            if len(sequence) < sequence_length:

                padding = np.zeros(
                    (
                        sequence_length - len(sequence),
                        len(feature_columns)
                    )
                )

                sequence = np.vstack(
                    [padding, sequence]
                )

            X_sequences.append(sequence)
            y_labels.append(labels[i])

    return (
        np.array(X_sequences, dtype=np.float32),
        np.array(y_labels, dtype=np.int32)
    )

In [8]:
X_train_seq_sample, y_train_seq_sample = create_sequences(
    train_sample,
    SEQUENCE_FEATURES,
    SEQUENCE_LENGTH
)

print("Sequence shape:", X_train_seq_sample.shape)
print("Label shape:", y_train_seq_sample.shape)

Sequence shape: (0,)
Label shape: (0,)


In [9]:
print(
    "Legitimate:",
    (y_train_seq_sample == 0).sum()
)

print(
    "Fraud:",
    (y_train_seq_sample == 1).sum()
)

Legitimate: 0
Fraud: 0


In [10]:
print("Unique users:", train_sample["nameOrig"].nunique())
print("Total rows:", len(train_sample))

user_counts = train_sample["nameOrig"].value_counts()

print("\nUsers with more than 1 transaction:")
print((user_counts > 1).sum())

print("\nMaximum transactions by one user:")
print(user_counts.max())

Unique users: 100000
Total rows: 100000

Users with more than 1 transaction:
0

Maximum transactions by one user:
1


In [11]:
print(
    train_sample[
        train_sample["nameOrig"].duplicated(keep=False)
    ][
        ["step", "nameOrig", "amount", "isFraud"]
    ].head(20)
)

Empty DataFrame
Columns: [step, nameOrig, amount, isFraud]
Index: []


In [12]:
chunksize = 100_000

for chunk_number, chunk in enumerate(
    pd.read_csv(TRAIN_FILE, chunksize=chunksize),
    start=1
):
    
    repeated_users = (
        chunk["nameOrig"].value_counts() > 1
    ).sum()

    if repeated_users > 0:
        print(
            "First chunk with repeated users:",
            chunk_number
        )
        print(
            "Repeated users in this chunk:",
            repeated_users
        )
        break

First chunk with repeated users: 3
Repeated users in this chunk: 5


In [13]:
seen_users = set()

first_repeat_chunk = None
repeat_count = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(TRAIN_FILE, chunksize=100_000),
    start=1
):
    
    current_users = set(chunk["nameOrig"])

    repeated = current_users.intersection(seen_users)

    if len(repeated) > 0:
        first_repeat_chunk = chunk_number
        repeat_count = len(repeated)

        print(
            "First chunk containing previously seen users:",
            chunk_number
        )

        print(
            "Previously seen users in this chunk:",
            repeat_count
        )

        break

    seen_users.update(current_users)

First chunk containing previously seen users: 2
Previously seen users in this chunk: 4


In [14]:
user_history = {}

In [15]:
from collections import defaultdict
import numpy as np
import pandas as pd

SEQUENCE_LENGTH = 5

user_history = {}

X_sequences = []
y_sequences = []

processed_rows = 0
sequence_count = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(TRAIN_FILE, chunksize=100_000),
    start=1
):
    
    # Keep chronological order
    chunk = chunk.sort_values("step")

    for row in chunk.itertuples(index=False):

        user = row.nameOrig

        # Current transaction's feature vector
        current_features = np.array(
            [
                getattr(row, feature)
                if pd.notna(getattr(row, feature))
                else 0.0
                for feature in SEQUENCE_FEATURES
            ],
            dtype=np.float32
        )

        # Get previous transactions for this user
        history = user_history.get(user, [])

        # Create sequence using ONLY previous transactions
        sequence = history[-SEQUENCE_LENGTH:]

        # Zero padding
        if len(sequence) < SEQUENCE_LENGTH:

            padding = [
                np.zeros(
                    len(SEQUENCE_FEATURES),
                    dtype=np.float32
                )
                for _ in range(
                    SEQUENCE_LENGTH - len(sequence)
                )
            ]

            sequence = padding + sequence

        X_sequences.append(sequence)

        # Current transaction is the label
        y_sequences.append(int(row.isFraud))

        # Add current transaction to history AFTER creating sequence
        history.append(current_features)

        user_history[user] = history[-SEQUENCE_LENGTH:]

        sequence_count += 1

    processed_rows += len(chunk)

    print(
        f"Chunk {chunk_number}/"
        f"{6362620 // 100000 + 1} processed | "
        f"Rows: {processed_rows:,} | "
        f"Sequences: {sequence_count:,}"
    )

Chunk 1/64 processed | Rows: 100,000 | Sequences: 100,000
Chunk 2/64 processed | Rows: 200,000 | Sequences: 200,000
Chunk 3/64 processed | Rows: 300,000 | Sequences: 300,000
Chunk 4/64 processed | Rows: 400,000 | Sequences: 400,000
Chunk 5/64 processed | Rows: 500,000 | Sequences: 500,000
Chunk 6/64 processed | Rows: 600,000 | Sequences: 600,000
Chunk 7/64 processed | Rows: 700,000 | Sequences: 700,000
Chunk 8/64 processed | Rows: 800,000 | Sequences: 800,000
Chunk 9/64 processed | Rows: 900,000 | Sequences: 900,000
Chunk 10/64 processed | Rows: 1,000,000 | Sequences: 1,000,000
Chunk 11/64 processed | Rows: 1,100,000 | Sequences: 1,100,000
Chunk 12/64 processed | Rows: 1,200,000 | Sequences: 1,200,000
Chunk 13/64 processed | Rows: 1,300,000 | Sequences: 1,300,000
Chunk 14/64 processed | Rows: 1,400,000 | Sequences: 1,400,000
Chunk 15/64 processed | Rows: 1,500,000 | Sequences: 1,500,000
Chunk 16/64 processed | Rows: 1,600,000 | Sequences: 1,600,000
Chunk 17/64 processed | Rows: 1,700,0

In [16]:
pd.read_csv(
    TRAIN_FILE,
    chunksize=100_000,
    nrows=500_000
)

In [17]:
user_history = {}

X_sequences = []
y_sequences = []

processed_rows = 0
sequence_count = 0

print("Sequence generation reset.")

Sequence generation reset.


In [18]:
X_sequences = np.array(
    X_sequences,
    dtype=np.float32
)

y_sequences = np.array(
    y_sequences,
    dtype=np.int32
)

print("X sequence shape:", X_sequences.shape)
print("y sequence shape:", y_sequences.shape)

print(
    "Legitimate:",
    (y_sequences == 0).sum()
)

print(
    "Fraud:",
    (y_sequences == 1).sum()
)

X sequence shape: (0,)
y sequence shape: (0,)
Legitimate: 0
Fraud: 0


In [19]:
user_history = {}

X_sequences = []
y_sequences = []

processed_rows = 0
sequence_count = 0

print("Sequence generation reset.")

Sequence generation reset.


In [20]:
for chunk_number, chunk in enumerate(
    pd.read_csv(
        TRAIN_FILE,
        chunksize=100_000,
        nrows=500_000
    ),
    start=1
):

    chunk = chunk.sort_values("step")

    for row in chunk.itertuples(index=False):

        user = row.nameOrig

        history = user_history.get(user, [])

        current_features = np.array(
            [
                getattr(row, feature)
                if pd.notna(getattr(row, feature))
                else 0.0
                for feature in SEQUENCE_FEATURES
            ],
            dtype=np.float32
        )

        sequence = history[-SEQUENCE_LENGTH:]

        while len(sequence) < SEQUENCE_LENGTH:
            sequence.insert(
                0,
                np.zeros(
                    len(SEQUENCE_FEATURES),
                    dtype=np.float32
                )
            )

        X_sequences.append(sequence)
        y_sequences.append(int(row.isFraud))

        history.append(current_features)

        user_history[user] = history[-SEQUENCE_LENGTH:]

    processed_rows += len(chunk)

    print(
        f"Chunk {chunk_number}/5 completed | "
        f"Rows: {processed_rows:,}"
    )

Chunk 1/5 completed | Rows: 100,000
Chunk 2/5 completed | Rows: 200,000
Chunk 3/5 completed | Rows: 300,000
Chunk 4/5 completed | Rows: 400,000
Chunk 5/5 completed | Rows: 500,000


In [21]:
X_sequences = np.array(
    X_sequences,
    dtype=np.float32
)

y_sequences = np.array(
    y_sequences,
    dtype=np.int32
)

print("X sequence shape:", X_sequences.shape)
print("y sequence shape:", y_sequences.shape)

print(
    "Legitimate:",
    (y_sequences == 0).sum()
)

print(
    "Fraud:",
    (y_sequences == 1).sum()
)

X sequence shape: (500000, 5, 19)
y sequence shape: (500000,)
Legitimate: 499767
Fraud: 233


In [22]:
from sklearn.preprocessing import StandardScaler

n_samples, sequence_length, n_features = X_sequences.shape

scaler = StandardScaler()

X_sequences_2d = X_sequences.reshape(
    -1,
    n_features
)

X_sequences_scaled_2d = scaler.fit_transform(
    X_sequences_2d
)

X_sequences_scaled = X_sequences_scaled_2d.reshape(
    n_samples,
    sequence_length,
    n_features
)

print("Scaled sequence shape:", X_sequences_scaled.shape)

Scaled sequence shape: (500000, 5, 19)


In [23]:
print(
    "Mean:",
    X_sequences_scaled.mean()
)

print(
    "Std:",
    X_sequences_scaled.std()
)

Mean: 8.223684e-11
Std: 0.7608859


In [24]:
feature_means = X_sequences_scaled.mean(axis=(0, 1))
feature_stds = X_sequences_scaled.std(axis=(0, 1))

for i, feature in enumerate(SEQUENCE_FEATURES):
    print(
        f"{feature:40s} "
        f"mean={feature_means[i]:.4f} "
        f"std={feature_stds[i]:.4f}"
    )


amount                                   mean=0.0000 std=1.0000
oldbalanceOrg                            mean=-0.0000 std=1.0000
newbalanceOrig                           mean=-0.0000 std=1.0000
oldbalanceDest                           mean=-0.0000 std=1.0000
newbalanceDest                           mean=-0.0000 std=1.0000
user_transaction_count_before            mean=0.0000 std=0.0000
previous_transaction_amount              mean=0.0000 std=0.0000
time_since_previous_transaction          mean=0.0000 std=0.0000
previous_average_amount                  mean=0.0000 std=0.0000
amount_deviation                         mean=0.0000 std=0.0000
amount_to_previous_average               mean=0.0000 std=0.0000
balance_depletion                        mean=-0.0000 std=1.0000
amount_to_balance_ratio                  mean=-0.0000 std=1.0000
receiver_transaction_count_before        mean=0.0000 std=1.0000
receiver_previous_amount                 mean=-0.0000 std=1.0000
receiver_transaction_frequency   

In [25]:
negative_count = (y_sequences == 0).sum()
positive_count = (y_sequences == 1).sum()

gru_scale_pos_weight = (
    negative_count / positive_count
)

print("Negative:", negative_count)
print("Positive:", positive_count)
print("GRU scale_pos_weight:", gru_scale_pos_weight)

Negative: 499767
Positive: 233
GRU scale_pos_weight: 2144.922746781116


In [26]:
split_index = int(len(X_sequences_scaled) * 0.8)

X_gru_train = X_sequences_scaled[:split_index]
y_gru_train = y_sequences[:split_index]

X_gru_val = X_sequences_scaled[split_index:]
y_gru_val = y_sequences[split_index:]

print("GRU train:", X_gru_train.shape)
print("GRU validation:", X_gru_val.shape)

print("Train fraud:", (y_gru_train == 1).sum())
print("Validation fraud:", (y_gru_val == 1).sum())

GRU train: (400000, 5, 19)
GRU validation: (100000, 5, 19)
Train fraud: 206
Validation fraud: 27


In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

gru_model = Sequential([
    GRU(
        64,
        input_shape=(SEQUENCE_LENGTH, len(SEQUENCE_FEATURES))
    ),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(
            name="roc_auc"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        ),
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        )
    ]
)

gru_model.summary()

c:\fraud detection system\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        16,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,433 (72.00 KB)

 Trainable params: 18,433 (72.00 KB)

 Non-trainable params: 0 (0.00 B)

In [28]:
gru_class_weight = {
    0: 1.0,
    1: gru_scale_pos_weight
}

print(gru_class_weight)

{0: 1.0, 1: np.float64(2144.922746781116)}


In [29]:
early_stopping = EarlyStopping(
    monitor="val_pr_auc",
    mode="max",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

In [30]:
history = gru_model.fit(
    X_gru_train,
    y_gru_train,

    validation_data=(
        X_gru_val,
        y_gru_val
    ),

    epochs=10,
    batch_size=512,

    class_weight=gru_class_weight,

    callbacks=[
        early_stopping
    ],

    verbose=1
)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 1.4644 - pr_auc: 4.6428e-04 - precision: 4.9701e-04 - recall: 0.5825 - roc_auc: 0.4764 - val_loss: 0.7225 - val_pr_auc: 2.7005e-04 - val_precision: 2.7005e-04 - val_recall: 1.0000 - val_roc_auc: 0.5001
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 1.4586 - pr_auc: 4.4542e-04 - precision: 5.1252e-04 - recall: 0.9854 - roc_auc: 0.4617 - val_loss: 0.7134 - val_pr_auc: 2.7005e-04 - val_precision: 2.7005e-04 - val_recall: 1.0000 - val_roc_auc: 0.5001
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 1.4598 - pr_auc: 4.8427e-04 - precision: 5.2351e-04 - recall: 0.8932 - roc_auc: 0.4836 - val_loss: 0.7310 - val_pr_auc: 2.7005e-04 - val_precision: 2.7005e-04 - val_recall: 1.0000 - val_roc_auc: 0.5001
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [31]:
gru_val_probability = gru_model.predict(
    X_gru_val,
    batch_size=1024
).ravel()

print(
    "GRU validation probability range:",
    gru_val_probability.min(),
    "to",
    gru_val_probability.max()
)

98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
GRU validation probability range: 0.03338589 to 0.5145068


In [32]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

gru_pr_auc = average_precision_score(
    y_gru_val,
    gru_val_probability
)

gru_roc_auc = roc_auc_score(
    y_gru_val,
    gru_val_probability
)

print("GRU Validation PR-AUC:", gru_pr_auc)
print("GRU Validation ROC-AUC:", gru_roc_auc)

GRU Validation PR-AUC: 0.0002700459078043267
GRU Validation ROC-AUC: 0.5000850229561982


In [33]:
from sklearn.metrics import classification_report

gru_val_predictions = (
    gru_val_probability >= 0.5
).astype(int)

print(
    classification_report(
        y_gru_val,
        gru_val_predictions,
        target_names=["Legitimate", "Fraud"],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

  Legitimate     1.0000    0.0002    0.0003     99973
       Fraud     0.0003    1.0000    0.0005        27

    accuracy                         0.0004    100000
   macro avg     0.5001    0.5001    0.0004    100000
weighted avg     0.9997    0.0004    0.0003    100000



In [34]:
print("Training fraud:", (y_gru_train == 1).sum())
print("Validation fraud:", (y_gru_val == 1).sum())

print("\nPrediction statistics:")
print("Min:", gru_val_probability.min())
print("Max:", gru_val_probability.max())
print("Mean:", gru_val_probability.mean())
print("Median:", np.median(gru_val_probability))

Training fraud: 206
Validation fraud: 27

Prediction statistics:
Min: 0.03338589
Max: 0.5145068
Mean: 0.51444054
Median: 0.5145068


In [35]:
print(
    "Average probability - Legitimate:",
    gru_val_probability[y_gru_val == 0].mean()
)

print(
    "Average probability - Fraud:",
    gru_val_probability[y_gru_val == 1].mean()
)

Average probability - Legitimate: 0.5144405
Average probability - Fraud: 0.5145069
